# D2.7 · Stop authority

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.6 · Post-incident change surface](https://spbreed.github.io/cyber-commons/lessons/D2.6.html)**.

| | |
|---|---|
| Tools used | kagent |

## What this lesson is

**What it covers.** Time your own stop authority end to end.

**Why a security engineer needs it.** Nobody has rehearsed halting an autonomous workflow. The control it builds is: named holder, measured time-to-stop, tested.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

At three in the morning, the question is not what went wrong. It is who is allowed to stop it, on what evidence, without waiting for a forty-person bridge call to reach consensus.

> **At CyberTravels.** At three in the morning, who is allowed to stop all four agents without waiting for a bridge call? Pre-agreed authority beats consensus every time, and R1 is what happens while you wait.

## 2 · The framework

```
   03:00, the agent is acting, the evidence is partial

   who may say stop?          +-----------------------------+
                              | named role, on call         |
   on what evidence?          | pre-agreed trigger list     |
   what does stop mean?       | revoke + gateway cut        |
   who is told after?         | named, not assembled at 3am |
                              +-----------------------------+

   pre-agreed authority beats a forty-person bridge call
```

Stop authority is the control everyone assumes exists and almost nobody has
timed.

Five questions decide whether you have it, and each needs a name or a number
rather than an intention:

1. **Who** can halt an agent fleet without seeking approval?
2. **What** is the mechanism — and is it revocation, which survives a restart,
   or process termination, which does not?
3. **How long** does it take, measured end to end, not estimated?
4. **What breaks** when it fires — and has the business already agreed to that?
5. **Who turns it back on**, and against what evidence?

An untested stop button is a belief. The purpose of this lesson is to convert it
into a measurement, because the measurement is what an auditor, a regulator and
a board will each ask for in different words.

## 3 · The procedure, as a skill

The skill prints the vague answers beside the concrete ones, then establishes what each mechanism actually survives — killing the process does not survive a restart, revoking the identity does — and reports a measured twelve-second time-to-stop from a game day rather than an estimate.

### The skill — [`skills/response/stop-authority-readiness/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/stop-authority-readiness/SKILL.md)

```yaml
name: stop-authority-readiness
description: >-
  Turn "we can stop it" into a named mechanism with an owner, a measured
  time-to-stop from a game day, and a stated cost of stopping. Use to answer how
  long it would actually take to stop a running agent, before granting autonomy,
  and whenever stop authority has never been exercised.
allowed-tools: Read, Grep, Glob
```

# Killing the process does not survive a restart

Stop authority is usually described rather than specified: somebody can stop it,
probably quickly, and nobody has tried. Three questions turn that into a control
— which mechanism, who may invoke it without asking, and how long it takes when
measured rather than estimated.

## When to use this

Before raising an agent's autonomy, at any authorisation to run unattended, and
once a year as a game day.

## Procedure

**1 — Write the vague answers down and then the concrete ones.** Side by side.
"Ops can kill it" against "the on-call SRE revokes the workload identity in the
identity console, and here is the runbook". The contrast is what gets the work
scheduled.

**2 — Enumerate mechanisms and what each survives.** Killing the process does not
survive a restart or a scheduler. Revoking the identity does. Blocking egress
stops the effect and not the run. Record what each actually stops.

**3 — Name who may invoke it without asking.** Stop authority that needs an
approval is not stop authority; it is an escalation. If nobody may act alone,
that is the finding.

**4 — Run a game day and measure.** From decision to the agent being unable to
act. Report seconds. An estimate is not a measurement and this is the number
that is always wrong in the optimistic direction.

**5 — Cost the stop.** What stopping costs per minute in halted legitimate work.
Somebody will ask, and having the number is what makes the decision fast during
an incident.

## Example

**Input** — the fixture committed at the top of [`scripts/stop_authority_readiness.py`](scripts/stop_authority_readiness.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
who         VAGUE    the security team
            CONCRETE on-call SRE, no approval required for non-human identities

mechanism   VAGUE    we can turn off the agents
            CONCRETE revoke the SPIFFE identity at the gateway (survives restart)

time        VAGUE    quickly
            CONCRETE measured 12s decision→first failed call, game day 2026-07-04
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "answers": [{"question": "str", "vague": "str", "concrete": "str"}],
  "mechanisms": [{"name": "str", "stops": ["str"], "survives_restart": false}],
  "authority": {"may_invoke_alone": ["str"], "approval_required": false},
  "game_day": {"measured_seconds": 0, "estimated_seconds": 0},
  "cost_per_minute": 0.0,
  "ready": false
}
```

## Failure modes

- **Process termination as the mechanism.** The scheduler restarts it.
- **An estimated time-to-stop.** Measure it once and the estimate is revealed.
- **Stop authority behind an approval.** That is escalation with a different
  name.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/stop-authority-readiness/scripts/stop_authority_readiness.py
SCRIPT = "skills/response/stop-authority-readiness/scripts/stop_authority_readiness.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The vague and concrete answers print side by side. Killing the process stops the agent but does not survive a restart, while identity revocation does. The game-day timeline gives a measured 12-second time-to-stop, permitting 12 to 240 further actions depending on rate. The readiness check fails the vague version on three counts and passes the tested one.

## Your turn

Run the game day. The deliverable is the number, and the number is what goes in the evidence pack for E1.7 and the board slide for E3.5. An untested stop button is a belief.

---

**Next → [D2.8 · Regulatory clock](https://spbreed.github.io/cyber-commons/lessons/D2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*